# Decorators in Python — wrap a function with extra behavior

> **Explain it like I am five:** A plain gift is still the same gift after we add wrapping paper and a ribbon. A decorator wraps a function with extra behavior without rewriting the function's main job.

This notebook builds decorators slowly: function references → functions inside functions → closures → decorators → decorators with arguments.

## Learning goals

- understand that functions are objects;
- pass functions into other functions and return them;
- understand closures;
- translate `@decorator` into normal function assignment;
- preserve arguments, return values, and metadata;
- build practical decorators safely.


## 1. Three ideas we need first

The original lesson starts with:


In [1]:
### function copy
### closures
### decorators


A more precise wording is:

1. **Function reference:** another name can point to the same function object.
2. **Closure:** an inner function remembers names from the enclosing function.
3. **Decorator:** a callable receives a function and returns a replacement callable.


## 2. Functions are objects

We can store a function in a variable just like a number or string.

**Original example:**


In [2]:
## function copy
def welcome():
    return "Welcome to the advanced python course"

welcome()


'Welcome to the advanced python course'

In [3]:
wel=welcome
print(wel())
del welcome
print(wel())


Welcome to the advanced python course
Welcome to the advanced python course


`wel = welcome` does not copy the function's code. Both names point to the same function object. Deleting the name `welcome` does not destroy the function because `wel` still points to it.

Notice the important difference:

- `wel` means the function object;
- `wel()` means call the function now.


In [4]:
print(wel)
print(wel())
print(callable(wel))


<function welcome at 0x000001E50340EF00>
Welcome to the advanced python course
True


## 3. Inner functions and the original closure example

A function may define another function. In the original example, the inner function can read `msg` from the outer function.


In [5]:
##closures functions

def main_welcome(msg):

    def sub_welcome_method():
        print("Welcome to the advance python course")
        print(msg)
        print("Please learn these concepts properly")
    return sub_welcome_method()


In [6]:
main_welcome("Welcome everyone")


Welcome to the advance python course
Welcome everyone
Please learn these concepts properly


### Important detail

The original code says `return sub_welcome_method()`, with parentheses. That **calls the inner function immediately** and returns its result (`None`). A true closure usually returns the function itself without parentheses so it can run later.


In [7]:
def make_welcome(msg):
    def sub_welcome_method():
        print("Welcome to the advanced Python course")
        print(msg)  # Remembered from make_welcome
        print("Please learn these concepts properly")
    return sub_welcome_method  # No parentheses: return the function.


saved_welcome = make_welcome("Welcome everyone")
print(saved_welcome)
saved_welcome()


<function make_welcome.<locals>.sub_welcome_method at 0x000001E50340F1C0>
Welcome to the advanced Python course
Welcome everyone
Please learn these concepts properly


That remembered `msg` is the **closure**. Even after `make_welcome` finishes, `saved_welcome` keeps the value it needs.


## 4. Pass a function into another function

A higher-order function accepts or returns another function. The next two original examples pass the built-in functions `print` and `len`.


In [8]:
def main_welcome(func):

    def sub_welcome_method():
        print("Welcome to the advance python course")
        func("Welcome everyone to this tutorial")
        print("Please learn these concepts properly")
    return sub_welcome_method()


In [9]:
main_welcome(print)


Welcome to the advance python course
Welcome everyone to this tutorial
Please learn these concepts properly


In [10]:
def main_welcome(func,lst):

    def sub_welcome_method():
        print("Welcome to the advance python course")
        print(func(lst))
        print("Please learn these concepts properly")
    return sub_welcome_method()


In [11]:
main_welcome(len,[1,2,3,4,5])


Welcome to the advance python course
5
Please learn these concepts properly


In [12]:
len([1,2,3,4,5,6])


6

## 5. Build the first decorator manually

A decorator normally returns a wrapper function. The wrapper runs extra code before and/or after the original function.


In [13]:
### Decorator
def main_welcome(func):

    def sub_welcome_method():
        print("Welcome to the advance python course")
        func()
        print("Please learn these concepts properly")
    return sub_welcome_method


In [14]:
def coure_introduction():
    print("This is an advanced python course")

coure_introduction()


This is an advanced python course


In [15]:
main_welcome(coure_introduction)


<function __main__.main_welcome.<locals>.sub_welcome_method()>

The last cell returns a wrapper but does not call it. That is why you see a function representation instead of the welcome messages. Store and call the result:


In [16]:
decorated_course = main_welcome(coure_introduction)
decorated_course()


Welcome to the advance python course
This is an advanced python course
Please learn these concepts properly


## 6. The `@` syntax

These two forms mean the same thing:

```python
@main_welcome
def course_introduction():
    ...
```

```python
def course_introduction():
    ...
course_introduction = main_welcome(course_introduction)
```

The `@` form is simply easier to read.


In [17]:
@main_welcome
def coure_introduction():
    print("This is an advanced python course")


In [18]:
coure_introduction()


Welcome to the advance python course
This is an advanced python course
Please learn these concepts properly


## 7. The original before-and-after decorator


In [19]:
## Decorator

def my_decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()
        print("Something is happening after the function is called.")
    return wrapper


In [20]:
@my_decorator
def say_hello():
    print("Hello!")


In [21]:
say_hello()


Something is happening before the function is called.
Hello!
Something is happening after the function is called.


Execution order is:

1. call `say_hello()` (which now refers to `wrapper`);
2. print the “before” message;
3. call the original `say_hello` kept in the closure as `func`;
4. print the “after” message.


## 8. A reusable decorator should accept arguments and return results

`*args` collects positional arguments and `**kwargs` collects keyword arguments. `return func(...)` sends the original result back to the caller.

`functools.wraps` preserves the original function's name and documentation.


In [22]:
from functools import wraps

def announce(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished.")
        return result
    return wrapper


@announce
def add(first, second=0):
    """Add two numbers."""
    return first + second


answer = add(4, second=6)
print("answer:", answer)
print("name:", add.__name__)
print("doc:", add.__doc__)


Calling add...
add finished.
answer: 10
name: add
doc: Add two numbers.


## 9. Decorators that accept their own arguments

The original `@repeat(3)` needs three layers:

1. `repeat(3)` remembers `n`;
2. `decorator(func)` receives the decorated function;
3. `wrapper(*args, **kwargs)` runs whenever the final function is called.


In [23]:
## Decorators WWith arguments
def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(n):
                func(*args, **kwargs)
        return wrapper
    return decorator


In [24]:
@repeat(3)
def say_hello():
    print("Hello")


In [25]:
say_hello()


Hello
Hello
Hello


Here is a production-friendlier version that preserves metadata and returns all repeated results:


In [26]:
def repeat_and_collect(times):
    if times < 0:
        raise ValueError("times must be zero or greater")

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]
        return wrapper
    return decorator


@repeat_and_collect(3)
def greet(name):
    return f"Hello, {name}!"


print(greet("Maya"))


['Hello, Maya!', 'Hello, Maya!', 'Hello, Maya!']


## 10. Practical example: validation

A decorator can protect several functions with the same rule.


In [27]:
def require_positive(func):
    @wraps(func)
    def wrapper(number, *args, **kwargs):
        if number <= 0:
            raise ValueError("number must be positive")
        return func(number, *args, **kwargs)
    return wrapper


@require_positive
def area_of_square(side):
    return side ** 2


print(area_of_square(5))

try:
    area_of_square(-2)
except ValueError as error:
    print("Validation stopped the call:", error)


25
Validation stopped the call: number must be positive


## 11. Stacking decorators

Decorators closest to the function are applied first. For:

```python
@outer
@inner
def work(): ...
```

Python builds `work = outer(inner(work))`. During a call, the outer wrapper begins first.


In [28]:
def label(name):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"enter {name}")
            result = func(*args, **kwargs)
            print(f"exit {name}")
            return result
        return wrapper
    return decorator


@label("OUTER")
@label("INNER")
def do_work():
    print("working")


do_work()


enter OUTER
enter INNER
working
exit INNER
exit OUTER


## 12. Decorating methods

The same `*args, **kwargs` pattern works for methods because the instance (`self`) is simply the first positional argument.


In [29]:
class Calculator:
    @announce
    def multiply(self, first, second):
        return first * second


calculator = Calculator()
print(calculator.multiply(6, 7))


Calling multiply...
multiply finished.
42


## 13. Common mistakes

- **Writing `@decorator()` when no configuration is expected:** use `@decorator`; parentheses mean “call a decorator factory.”
- **Forgetting `return wrapper`:** the decorated name becomes `None`.
- **Forgetting `return func(...)`:** the original result is lost.
- **Using a wrapper with no parameters:** decorated functions with arguments fail. Use `*args, **kwargs` for a general wrapper.
- **Forgetting `@wraps(func)`:** debugging and documentation show the wrapper's name instead of the original function's name.
- **Catching every exception inside a decorator:** this can hide real bugs. Catch only exceptions you can handle meaningfully.
- **Putting changing state in a closure without care:** that state is shared across calls to the same decorated function.


## 14. Mini practice

Predict the order of output, then run it.


In [30]:
def uppercase_result(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


@uppercase_result
def welcome_person(name):
    return f"Welcome, {name}"


print(welcome_person("Ravi"))


WELCOME, RAVI


## Easy revision cheat sheet

```python
from functools import wraps

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper

@decorator
def target(...):
    ...
```

| Idea | Remember |
|---|---|
| Function reference | `alias = function` has no parentheses |
| Call a function | `function()` uses parentheses |
| Closure | Inner function remembers enclosing values |
| Decoration | `target = decorator(target)` |
| `@decorator` | Cleaner spelling of the assignment above |
| Flexible wrapper | Use `*args, **kwargs` |
| Preserve result | `return func(*args, **kwargs)` |
| Preserve identity | Use `@wraps(func)` |
| Configured decorator | `@repeat(3)` needs three nested functions |

**One-sentence memory trick:** A decorator takes a function, puts it inside a wrapper, and returns the wrapper under the original name.

## Conclusion

Decorators are best for behavior that many functions share—logging, timing, validation, authorization, caching, retries, or registration. Keep wrappers small, preserve metadata, forward every argument, and return the original result unless changing it is the decorator's explicit purpose.
